In [ ]:
import numpy as np
import pandas as pd
from pickle import dump

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

import mlacca_functions as mlcf

In [ ]:
df_path = 'data/csv_files/new_vk_areas_df.csv'
df_with_pahs_path = 'data/csv_files/new_vk_areas_with_pahs.csv'

weighted = True

In [3]:
df = pd.read_csv(df_path)
df_with_pahs = pd.read_csv(df_with_pahs_path)

In [4]:
columns_2d = ['O/C','H/C']
X_2d = df[columns_2d].to_numpy()

columns_3d = ['O/C','H/C', 'N/C']
X_3d = df[columns_3d].to_numpy()
X_3d_scaled = StandardScaler().fit_transform(X_3d)

y = df['category'].to_numpy()
weights = df['weights'].to_numpy() if weighted else None


X_2d_with_pahs = df_with_pahs[columns_2d].to_numpy()
X_3d_with_pahs = df_with_pahs[columns_3d].to_numpy()
X_3d_with_pahs_scaled = StandardScaler().fit_transform(X_3d_with_pahs)

y_with_pahs = df_with_pahs['category'].to_numpy()

weights_with_pahs = df_with_pahs['weights'].to_numpy() if weighted else None

In [ ]:
def find_best_hyperparams(clf,params_dictionary,X,y,sample_weights=None,refit=True, verbose=3, scoring='balanced_accuracy'):
    grid = GridSearchCV(clf,params_dictionary, refit=refit, verbose=verbose, scoring=scoring)
    grid.fit(**dict(X=X,y=y,sample_weights=sample_weights) if sample_weights is not None else dict(X=X,y=y))
    return grid

## Optimise KNN

In [56]:
param_grid_knn = {
    'weights':['distance'],
    'n_neighbors': [1,5,10,50,100,150,200],
    }

In [ ]:
grid_knn_3d = find_best_hyperparams(KNeighborsClassifier(),param_grid_knn,X_3d_scaled,y)
print('3D KNN best_estimator_:', grid_knn_3d.best_estimator_)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
[CV 1/5] END ...n_neighbors=1, weights=distance;, score=0.920 total time=   0.0s
[CV 2/5] END ...n_neighbors=1, weights=distance;, score=0.831 total time=   0.0s
[CV 3/5] END ...n_neighbors=1, weights=distance;, score=0.887 total time=   0.0s
[CV 4/5] END ...n_neighbors=1, weights=distance;, score=0.676 total time=   0.0s
[CV 5/5] END ...n_neighbors=1, weights=distance;, score=0.762 total time=   0.0s
[CV 1/5] END ...n_neighbors=5, weights=distance;, score=0.915 total time=   0.0s
[CV 2/5] END ...n_neighbors=5, weights=distance;, score=0.903 total time=   0.0s
[CV 3/5] END ...n_neighbors=5, weights=distance;, score=0.891 total time=   0.0s
[CV 4/5] END ...n_neighbors=5, weights=distance;, score=0.696 total time=   0.0s
[CV 5/5] END ...n_neighbors=5, weights=distance;, score=0.806 total time=   0.0s
[CV 1/5] END ..n_neighbors=10, weights=distance;, score=0.949 total time=   0.0s
[CV 2/5] END ..n_neighbors=10, weights=distance;,

In [58]:
grid_knn_pahs_3d = find_best_hyperparams(KNeighborsClassifier(),param_grid_knn,X_3d_with_pahs_scaled,y_with_pahs)
print('3D KNN with PAHs best_estimator_:', grid_knn_pahs_3d.best_estimator_)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
[CV 1/5] END ...n_neighbors=1, weights=distance;, score=0.764 total time=   0.0s
[CV 2/5] END ...n_neighbors=1, weights=distance;, score=0.804 total time=   0.0s
[CV 3/5] END ...n_neighbors=1, weights=distance;, score=0.827 total time=   0.0s
[CV 4/5] END ...n_neighbors=1, weights=distance;, score=0.915 total time=   0.0s
[CV 5/5] END ...n_neighbors=1, weights=distance;, score=0.723 total time=   0.0s
[CV 1/5] END ...n_neighbors=5, weights=distance;, score=0.832 total time=   0.0s
[CV 2/5] END ...n_neighbors=5, weights=distance;, score=0.870 total time=   0.0s
[CV 3/5] END ...n_neighbors=5, weights=distance;, score=0.856 total time=   0.0s
[CV 4/5] END ...n_neighbors=5, weights=distance;, score=0.942 total time=   0.0s
[CV 5/5] END ...n_neighbors=5, weights=distance;, score=0.792 total time=   0.0s
[CV 1/5] END ..n_neighbors=10, weights=distance;, score=0.850 total time=   0.0s
[CV 2/5] END ..n_neighbors=10, weights=distance;,

## Optimising RBF SVM
The usual `GridSearchCV` doesn't seem to work properly for RBF: it outputs values of 100,000 for C, which is patently nonsensical for a dataset of <10,000 observations. Therefore, another – unfortunately slower – approach is taken.

In [ ]:
param_grid_svm_rbf = {
    'kernel':['rbf'],
    'C': [10**(x) for x in range(4)],
    'gamma': ['auto'] + [10**(-x) for x in range(5)],
    }
param_grid_svm_rbf

{'kernel': ['rbf'], 'C': [1, 10, 100, 1000], 'gamma': ['auto', 1, 0.1, 0.01, 0.001, 0.0001]}


In [ ]:
grid_rbf_svm_3d=find_best_hyperparams(SVC(),param_grid_svm_rbf,X_3d_scaled,y,sample_weight=weights)
print('3D RBF SVM best_estimator_:', grid_rbf_svm_3d.best_estimator_)


Fitting 5 folds for each of 24 candidates, totalling 120 fits
[CV 1/5] END .......C=1, gamma=auto, kernel=rbf;, score=0.951 total time=   0.0s
[CV 2/5] END .......C=1, gamma=auto, kernel=rbf;, score=0.939 total time=   0.0s
[CV 3/5] END .......C=1, gamma=auto, kernel=rbf;, score=0.949 total time=   0.0s
[CV 4/5] END .......C=1, gamma=auto, kernel=rbf;, score=0.788 total time=   0.0s
[CV 5/5] END .......C=1, gamma=auto, kernel=rbf;, score=0.904 total time=   0.0s
[CV 1/5] END ..........C=1, gamma=1, kernel=rbf;, score=0.953 total time=   0.1s
[CV 2/5] END ..........C=1, gamma=1, kernel=rbf;, score=0.893 total time=   0.0s
[CV 3/5] END ..........C=1, gamma=1, kernel=rbf;, score=0.946 total time=   0.0s
[CV 4/5] END ..........C=1, gamma=1, kernel=rbf;, score=0.786 total time=   0.0s
[CV 5/5] END ..........C=1, gamma=1, kernel=rbf;, score=0.862 total time=   0.0s
[CV 1/5] END ........C=1, gamma=0.1, kernel=rbf;, score=0.950 total time=   0.0s
[CV 2/5] END ........C=1, gamma=0.1, kernel=rbf

In [ ]:
# print('3D SVM (RBF) with PAHs best_estimator_', rbf_opt(param_grid_svm_rbf,X_3d_with_pahs,y_with_pahs))
grid_rbf_svm_pahs_3d = find_best_hyperparams(SVC(),param_grid_svm_rbf,X_3d_with_pahs_scaled,y_with_pahs,sample_weights=weights_with_pahs)
print('3D RBF SVM with PAHs best_estimator_:', grid_rbf_svm_pahs_3d.best_estimator_)

## Optimising Polynomial SVM

In [ ]:
# choose C and gamma hyperparameters for SVM with polynomial kernel for the 3D case
# https://www.geeksforgeeks.org/machine-learning/svm-hyperparameter-tuning-using-gridsearchcv-ml/

param_grid_svm_poly = {
    'kernel':['poly'],
    'degree': list(range(1,5)),
    'C': [10**x for x in range(4)],
    'gamma': ['auto'] + [10**(-x) for x in range(5)],
    }
param_grid_svm_poly

{'degree': [1, 2, 3, 4],
 'C': [1, 10, 100, 1000],
 'gamma': ['auto', 1, 0.1, 0.01, 0.001, 0.0001]}

In [ ]:
grid_poly_svm_3d = find_best_hyperparams(SVC(),param_grid_svm_poly,X_3d_scaled,y,sample_weights=weights)
print('3D poly SVM best_estimator_:', grid_poly_svm_3d.best_estimator_)

Fitting 5 folds for each of 96 candidates, totalling 480 fits
[CV 1/5] END .........C=1, degree=1, gamma=auto;, score=0.950 total time=   0.0s
[CV 2/5] END .........C=1, degree=1, gamma=auto;, score=0.941 total time=   0.0s
[CV 3/5] END .........C=1, degree=1, gamma=auto;, score=0.932 total time=   0.0s
[CV 4/5] END .........C=1, degree=1, gamma=auto;, score=0.793 total time=   0.0s
[CV 5/5] END .........C=1, degree=1, gamma=auto;, score=0.935 total time=   0.0s
[CV 1/5] END ............C=1, degree=1, gamma=1;, score=0.955 total time=   0.0s
[CV 2/5] END ............C=1, degree=1, gamma=1;, score=0.933 total time=   0.0s
[CV 3/5] END ............C=1, degree=1, gamma=1;, score=0.945 total time=   0.0s
[CV 4/5] END ............C=1, degree=1, gamma=1;, score=0.777 total time=   0.0s
[CV 5/5] END ............C=1, degree=1, gamma=1;, score=0.935 total time=   0.0s
[CV 1/5] END ..........C=1, degree=1, gamma=0.1;, score=0.950 total time=   0.0s
[CV 2/5] END ..........C=1, degree=1, gamma=0.1

In [ ]:
grid_poly_svm_pahs_3d = find_best_hyperparams(SVC(),X_3d_with_pahs_scaled,y_with_pahs,y,sample_weights=weights_with_pahs)
print('3D poly SVM with PAHs best_estimator_:', grid_poly_svm_pahs_3d.best_estimator_)

Fitting 5 folds for each of 96 candidates, totalling 480 fits
[CV 1/5] END .........C=1, degree=1, gamma=auto;, score=0.937 total time=   0.1s
[CV 2/5] END .........C=1, degree=1, gamma=auto;, score=0.921 total time=   0.1s
[CV 3/5] END .........C=1, degree=1, gamma=auto;, score=0.972 total time=   0.2s
[CV 4/5] END .........C=1, degree=1, gamma=auto;, score=0.902 total time=   0.2s
[CV 5/5] END .........C=1, degree=1, gamma=auto;, score=0.898 total time=   0.2s
[CV 1/5] END ............C=1, degree=1, gamma=1;, score=0.938 total time=   0.3s
[CV 2/5] END ............C=1, degree=1, gamma=1;, score=0.914 total time=   0.1s
[CV 3/5] END ............C=1, degree=1, gamma=1;, score=0.972 total time=   0.2s
[CV 4/5] END ............C=1, degree=1, gamma=1;, score=0.893 total time=   0.3s
[CV 5/5] END ............C=1, degree=1, gamma=1;, score=0.902 total time=   0.2s
[CV 1/5] END ..........C=1, degree=1, gamma=0.1;, score=0.931 total time=   0.2s
[CV 2/5] END ..........C=1, degree=1, gamma=0.1

In [ ]:
text_file = open('data/hyperparameters.txt', 'w')

text_file.write(f'''3D KNN best_estimator_: {grid_knn_3d.best_estimator_}
3D KNN with PAHs best_estimator_: {grid_knn_pahs_3d.best_estimator_}

3D RBF SVM best_estimator_: {grid_rbf_svm_3d.best_estimator_}
3D RBF SVM with PAHs best_estimator_: {grid_rbf_svm_pahs_3d.best_estimator_}
                
3D SVM (poly) best_estimator_: {grid_poly_svm_3d.best_estimator_}
3D SVM (poly) with PAHs best_estimator_: {grid_poly_svm_pahs_3d.best_estimator_}
''')

text_file.close()

In [ ]:
hyperparameters = {
    'knn_3d':grid_knn_3d.best_params_,
    'knn_pahs_3d':grid_knn_pahs_3d.best_params_,
    'rbf_svm_3d':grid_rbf_svm_3d.best_params_,
    'rbf_svm_pahs_3d':grid_rbf_svm_pahs_3d.best_params_,
    'poly_svm_3d':grid_poly_svm_3d.best_params_,
    'poly_svm_pahs_3d':grid_poly_svm_pahs_3d.best_params_
}
with open(f'data/hyperparameters/hyperparameters.pkl', 'wb') as f:
    dump(hyperparameters, f)